In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

# 모델 관련
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, KFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# 언더샘플링
from imblearn.under_sampling import RandomUnderSampler

# 부스터 모델
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 딥러닝 (CNN)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# -------------------------------
# 0. 데이터 불러오기
X = pd.read_csv("X_mm.csv")
y = pd.read_csv("y.csv").squeeze()

# y 값 확인 및 변환 (혹시 모를 [1,2] → [0,1])
if set(y.unique()) == {1, 2}:
    y = y.map({1:0, 2:1}).astype(int)

print("✅ 데이터 로드 완료:", X.shape, y.shape, "Unique y:", y.unique())

# -------------------------------
# 1. 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba, fold):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Fold": fold,
        "Algorithm": name,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

# -------------------------------
# 2. 파라미터 후보 (최소화)
param_grids = {
    "Logistic Regression": {'C': [0.1, 1, 10]},
    "Random Forest": {'n_estimators': [100], 'max_depth': [5, 10]},
    "XGBoost": {'n_estimators': [100], 'max_depth': [3, 5]},
    "LightGBM": {'n_estimators': [100], 'num_leaves': [31, 50]},
    "CatBoost": {'depth': [4, 6], 'iterations': [100], 'learning_rate': [0.1]},
    "MLP": {'hidden_layer_sizes': [(64,), (64, 32)], 'alpha': [0.0001, 0.001]}
}

# -------------------------------
# 3. 기본 모델
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(max_iter=300, random_state=42)
}

# -------------------------------
# 4. 학습 + 평가 (5-Fold, 데이터 누수 방지)
results = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    # (1) train/valid 분리
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    # (2) 언더샘플링은 train에만 적용
    rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
    X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

    # (3) 스케일링: train에 fit → train/valid에 각각 transform
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train_res).astype(np.float32)
    X_valid_scaled = scaler.transform(X_valid).astype(np.float32)

    # (4) 전통 ML 모델
    for name, model in base_models.items():
        clf = RandomizedSearchCV(
            model,
            param_distributions=param_grids[name],
            n_iter=2, cv=3, scoring='f1',
            n_jobs=2, random_state=42
        )
        clf.fit(X_train_scaled, y_train_res)
        best_model = clf.best_estimator_
        y_pred = best_model.predict(X_valid_scaled)
        y_proba = best_model.predict_proba(X_valid_scaled)[:, 1]
        results.append(evaluate_model(name, y_valid, y_pred, y_proba, fold))

    # (5) CNN
    cnn = Sequential([
        Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    cnn.fit(X_train_scaled, y_train_res, epochs=5, batch_size=32, verbose=0)

    y_pred_cnn = cnn.predict(X_valid_scaled).flatten()
    y_pred_cnn_label = (y_pred_cnn >= 0.5).astype(int)
    results.append(evaluate_model("CNN", y_valid, y_pred_cnn_label, y_pred_cnn, fold))

# -------------------------------
# 5. KNN 평가 (언더샘플링 train만 적용)
X_scaled_all = MinMaxScaler().fit_transform(X).astype(np.float32)
X_res_all, y_res_all = RandomUnderSampler(sampling_strategy=0.5, random_state=42).fit_resample(X_scaled_all, y)

knn = KNeighborsClassifier(n_neighbors=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict')
y_proba_knn = cross_val_predict(knn, X_res_all, y_res_all, cv=kf, method='predict_proba')[:, 1]
results.append(evaluate_model("KNN", y_res_all, y_pred_knn, y_proba_knn, fold='All'))

# -------------------------------
# 6. 결과 정리 + 저장
result_df = pd.DataFrame(results)

summary = result_df.groupby("Algorithm")[[ 
    "Precision", "Recall", "Specificity", "F1-score", 
    "Accuracy", "Balanced Accuracy", "AUC"
]].mean().round(4).sort_values("AUC", ascending=False)

result_df.to_csv("all_fold_results.csv", index=False, encoding="utf-8-sig")
summary.to_csv("model_summary.csv", encoding="utf-8-sig")

print("📂 저장 완료")
print(" - all_fold_results.csv : Fold별 전체 성능 기록")
print(" - model_summary.csv : 모델별 평균 성능 요약")